In [1]:
from sklearn.preprocessing import StandardScaler
import pandas as pd
from sklearn.utils import shuffle
from sklearn.model_selection import train_test_split, GridSearchCV, RepeatedKFold
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis as QDA
from sklearn.ensemble import AdaBoostClassifier as AdaB
from sklearn.ensemble import BaggingClassifier as BAG
from sklearn.ensemble import GradientBoostingClassifier as GraB
from sklearn.ensemble import RandomForestClassifier as RF
from sklearn.linear_model import LogisticRegressionCV as LR
from sklearn.naive_bayes import GaussianNB as GNB
from sklearn.neighbors import KNeighborsClassifier as KNN
from sklearn.neural_network import MLPClassifier as NN
from sklearn import svm
from sklearn.tree import DecisionTreeClassifier as DT

In [2]:
path_data = r'...' # The path of the file which combines all extracted radiomic features with labels of endoleak and dataset
path_feature = r'...\FeatureReduction.xlsx'

In [3]:
def x_standard(x):
    colNames = x.columns
    x = x.astype(np.float64)
    x = StandardScaler().fit_transform(x) # 均值为0，方差为1
    x = pd.DataFrame(x)
    x.columns = colNames
    return x

In [4]:
feature = pd.read_excel(path_feature)
feature = feature.drop(columns = 'Unnamed: 0')
data = pd.read_excel(path_data)
data = data[feature.columns]
y = data['Endoleak']
x = data[data.columns[3:]]
x = x_standard(x)

In [5]:
data_train = data[data['Dataset'] == 'Train']
y_train = data_train['Endoleak']
x_train = data_train[data_train.columns[3:]]
x_train = x_standard(x_train)

In [6]:
data_test = data[data['Dataset'] == 'Test']
y_test = data_test['Endoleak']
x_test = data_test[data_test.columns[3:]]
x_test = x_standard(x_test)

In [7]:
def model_index(model_name, model, model_eva, x, y):
    y_proba = model.predict_proba(x)
    auc_score = roc_auc_score(y, y_proba[:,1])
    fpr, tpr, thresholds = roc_curve(y, y_proba[:,1], pos_label=1)
    index = np.argmax(tpr - fpr)
    threshold = thresholds[index]
    
    y_pred = y_proba[:,1] - threshold
    y_pred[y_pred >= 0] = 1
    y_pred[y_pred < 0] = 0
    cm = confusion_matrix(y, y_pred)
    TN = cm[0,0]; FP = cm[0,1]
    FN = cm[1,0]; TP = cm[1,1]
    SEN = TP / (TP+FN)
    SPE = TN / (TN+FP)
    FPR = FP / (TN+FP)
    FNR = FN / (TP+FN)
    PPV = TP / (TP+FP)
    NPV = TN / (TN+FN)
    ACC = (TP+TN) / (TP+TN+FP+FN)
    line = pd.Series({'AUC':auc_score, 'Threshold':threshold, 'SEN':SEN, 'SPE':SPE, 'FPR':FPR, 'FNR':FNR, 'PPV':PPV, 'NPV':NPV, 'ACC':ACC},
                     name=model_name)
    model_eva = model_eva.append(line, ignore_index=False)
    return model_eva, y_proba, auc_score, fpr, tpr, cm

In [8]:
# Adaptive boosting
param_grid = dict(n_estimators=range(1,101,1))
grid = GridSearchCV(AdaB(algorithm='SAMME', random_state=0), param_grid=param_grid, cv=10).fit(x_train, y_train)
n_estimators = grid.best_params_['n_estimators']
print('Estimator:', grid.best_estimator_)
model_eva = pd.DataFrame(columns = ['AUC', 'Threshold', 'SEN', 'SPE', 'FPR', 'FNR', 'PPV', 'NPV', 'ACC'])
model = AdaB(algorithm='SAMME', random_state=0, n_estimators=n_estimators).fit(x_train, y_train)
model_eva, y_proba, auc_score, fpr, tpr, cm = model_index('Train', model, model_eva, x_train, y_train)
model_eva, y_proba, auc_score, fpr, tpr, cm = model_index('Test', model, model_eva, x_test, y_test)
model_eva

In [9]:
# Bagging
param_grid = dict(random_state=range(0,21,1))
grid = GridSearchCV(BAG(), param_grid=param_grid, cv=10).fit(x_train, y_train)
random_state = grid.best_params_['random_state']
param_grid = dict(n_estimators=range(1,101,1))
grid = GridSearchCV(BAG(random_state=random_state), param_grid=param_grid, cv=10).fit(x_train, y_train)
n_estimators = grid.best_params_['n_estimators']
param_grid = dict(max_features=np.arange(1,16,1))
grid = GridSearchCV(BAG(random_state=random_state, n_estimators=n_estimators), param_grid=param_grid, cv=10).fit(x_train, y_train)
max_features = grid.best_params_['max_features']
print('Estimator:', grid.best_estimator_)
model_eva = pd.DataFrame(columns = ['AUC', 'Threshold', 'SEN', 'SPE', 'FPR', 'FNR', 'PPV', 'NPV', 'ACC'])
model = BAG(random_state=random_state, n_estimators=n_estimators, max_features=max_features).fit(x_train, y_train)
model_eva, y_proba, auc_score, fpr, tpr, cm = model_index('Train', model, model_eva, x_train, y_train)
model_eva, y_proba, auc_score, fpr, tpr, cm = model_index('Test', model, model_eva, x_test, y_test)
model_eva

In [10]:
# Decision tree
model_eva = pd.DataFrame(columns = ['AUC', 'Threshold', 'SEN', 'SPE', 'FPR', 'FNR', 'PPV', 'NPV', 'ACC'])
for i in range(100):
    model = DT(criterion='gini').fit(x_train, y_train)
    model_eva, y_proba, auc_score, fpr, tpr, cm = model_index('DT_Test', model, model_eva, x_test, y_test)
print(np.mean(model_eva['AUC']))
model_eva = pd.DataFrame(columns = ['AUC', 'Threshold', 'SEN', 'SPE', 'FPR', 'FNR', 'PPV', 'NPV', 'ACC'])
for i in range(100):
    model = DT(criterion='entropy').fit(x_train, y_train)
    model_eva, y_proba, auc_score, fpr, tpr, cm = model_index('DT_Test', model, model_eva, x_test, y_test)
print(np.mean(model_eva['AUC']))
param_grid = dict(max_features=np.arange(1,16,1))
grid = GridSearchCV(DT(criterion='gini', random_state=10), param_grid=param_grid, cv=10).fit(x_train, y_train)
max_features = grid.best_params_['max_features']
print('Estimator:', grid.best_estimator_)
model_eva = pd.DataFrame(columns = ['AUC', 'Threshold', 'SEN', 'SPE', 'FPR', 'FNR', 'PPV', 'NPV', 'ACC'])
model = DT(criterion='gini', random_state=10, max_features=max_features).fit(x_train, y_train)
model_eva, y_proba, auc_score, fpr, tpr, cm = model_index('Train', model, model_eva, x_train, y_train)
model_eva, y_proba, auc_score, fpr, tpr, cm = model_index('Test', model, model_eva, x_test, y_test)
model_eva

In [11]:
# Gradient boosting
param_grid = dict(random_state=range(0,21,1))
grid = GridSearchCV(GraB(), param_grid=param_grid, cv=10).fit(x_train, y_train)
random_state = grid.best_params_['random_state']
param_grid = dict(n_estimators=range(1,101,1))
grid = GridSearchCV(GraB(random_state=random_state), param_grid=param_grid, cv=10).fit(x_train, y_train)
n_estimators = grid.best_params_['n_estimators']
param_grid = dict(max_features=np.arange(1,16,1))
grid = GridSearchCV(GraB(random_state=random_state, n_estimators=n_estimators), param_grid=param_grid, cv=10).fit(x_train, y_train)
max_features = grid.best_params_['max_features']
print('Estimator:', grid.best_estimator_)
model_eva = pd.DataFrame(columns = ['AUC', 'Threshold', 'SEN', 'SPE', 'FPR', 'FNR', 'PPV', 'NPV', 'ACC'])
model = GraB(random_state=random_state, n_estimators=n_estimators, max_features=max_features).fit(x_train, y_train)
model_eva, y_proba, auc_score, fpr, tpr, cm = model_index('Train', model, model_eva, x_train, y_train)
model_eva, y_proba, auc_score, fpr, tpr, cm = model_index('Test', model, model_eva, x_test, y_test)
model_eva

In [12]:
# Gradient boosting
model_eva = pd.DataFrame(columns = ['AUC', 'Threshold', 'SEN', 'SPE', 'FPR', 'FNR', 'PPV', 'NPV', 'ACC'])
model = GNB().fit(x_train, y_train)
model_eva, y_proba, auc_score, fpr, tpr, cm = model_index('Train', model, model_eva, x_train, y_train)
model_eva, y_proba, auc_score, fpr, tpr, cm = model_index('Test', model, model_eva, x_test, y_test)
model_eva

In [13]:
# Gradient boosting
param_grid = {'n_neighbors': range(1,101,1)}
grid = GridSearchCV(KNN(weights='distance'), param_grid=param_grid, cv=10).fit(x_train, y_train)
n_neighbors = grid.best_params_['n_neighbors']
print('Estimator:', grid.best_estimator_)
model_eva = pd.DataFrame(columns = ['AUC', 'Threshold', 'SEN', 'SPE', 'FPR', 'FNR', 'PPV', 'NPV', 'ACC'])
model = KNN(weights='distance', n_neighbors=n_neighbors).fit(x_train, y_train)
model_eva, y_proba, auc_score, fpr, tpr, cm = model_index('Train', model, model_eva, x_train, y_train)
model_eva, y_proba, auc_score, fpr, tpr, cm = model_index('Test', model, model_eva, x_test, y_test)
model_eva

In [14]:
# Linear discriminant analysis
model_eva = pd.DataFrame(columns = ['AUC', 'Threshold', 'SEN', 'SPE', 'FPR', 'FNR', 'PPV', 'NPV', 'ACC'])
model = LDA(solver='lsqr', shrinkage='auto').fit(x_train, y_train)
model_eva, y_proba, auc_score, fpr, tpr, cm = model_index('Train', model, model_eva, x_train, y_train)
model_eva, y_proba, auc_score, fpr, tpr, cm = model_index('Test', model, model_eva, x_test, y_test)
model_eva

In [15]:
# Logistic regression
Cs = np.logspace(-4,1,100, base=10)
model_eva = pd.DataFrame(columns = ['AUC', 'Threshold', 'SEN', 'SPE', 'FPR', 'FNR', 'PPV', 'NPV', 'ACC'])
model = LR(Cs=Cs, solver='liblinear', penalty='l2', cv=10).fit(x_train, y_train)
model_eva, y_proba, auc_score, fpr, tpr, cm = model_index('Train', model, model_eva, x_train, y_train)
model_eva, y_proba, auc_score, fpr, tpr, cm = model_index('Test', model, model_eva, x_test, y_test)
model_eva

In [16]:
# Logistic regression
param_grid = dict(random_state=range(0,21,1))
grid = GridSearchCV(NN(max_iter=2000), param_grid=param_grid, cv=10, scoring='roc_auc').fit(x_train, y_train)
random_state = grid.best_params_['random_state']
print('Estimator:', grid.best_estimator_)
model_eva = pd.DataFrame(columns = ['AUC', 'Threshold', 'SEN', 'SPE', 'FPR', 'FNR', 'PPV', 'NPV', 'ACC'])
model = NN(max_iter=2000, random_state=random_state).fit(x_train, y_train)
model_eva, y_proba, auc_score, fpr, tpr, cm = model_index('NN_Train', model, model_eva, x_train, y_train)
model_eva, y_proba, auc_score, fpr, tpr, cm = model_index('NN_Test', model, model_eva, x_test, y_test)
model_eva

In [17]:
# Quadratic discriminant analysis
model_eva = pd.DataFrame(columns = ['AUC', 'Threshold', 'SEN', 'SPE', 'FPR', 'FNR', 'PPV', 'NPV', 'ACC'])
model = QDA().fit(x_train, y_train)
model_eva, y_proba, auc_score, fpr, tpr, cm = model_index('Train', model, model_eva, x_train, y_train)
model_eva, y_proba, auc_score, fpr, tpr, cm = model_index('Test', model, model_eva, x_test, y_test)
model_eva

In [18]:
# Random forest
param_grid = dict(random_state=range(0,21,1))
grid = GridSearchCV(RF(), param_grid=param_grid, cv=10).fit(x_train, y_train)
random_state = grid.best_params_['random_state']
param_grid = dict(n_estimators=range(1,101,1))
grid = GridSearchCV(RF(random_state=random_state), param_grid=param_grid, cv=10).fit(x_train, y_train)
n_estimators = grid.best_params_['n_estimators']
param_grid = dict(max_features=np.arange(1,16,1))
grid = GridSearchCV(RF(random_state=random_state, n_estimators=n_estimators), param_grid=param_grid, cv=10).fit(x_train, y_train)
max_features = grid.best_params_['max_features']
print('Estimator:', grid.best_estimator_)
model_eva = pd.DataFrame(columns = ['AUC', 'Threshold', 'SEN', 'SPE', 'FPR', 'FNR', 'PPV', 'NPV', 'ACC'])
model = RF(random_state=random_state, n_estimators=n_estimators, max_features=max_features).fit(x_train, y_train)
model_eva, y_proba, auc_score, fpr, tpr, cm = model_index('BAG_Train', model, model_eva, x_train, y_train)
model_eva, y_proba, auc_score, fpr, tpr, cm = model_index('BAG_Test', model, model_eva, x_test, y_test)
model_eva

In [19]:
# Support vector machine
Cs = np.logspace(-1,1,20, base=2)
gammas = np.logspace(-5,-3,20, base=2)
param_grid = dict(C=Cs, gamma=gammas)
grid = GridSearchCV(svm.SVC(kernel='rbf', random_state=0), param_grid=param_grid, cv=10).fit(x_train, y_train)
C = grid.best_params_['C']
gamma = grid.best_params_['gamma']
print('Estimator:', grid.best_estimator_)
model_eva = pd.DataFrame(columns = ['AUC', 'Threshold', 'SEN', 'SPE', 'FPR', 'FNR', 'PPV', 'NPV', 'ACC'])
model = svm.SVC(probability=True, kernel='rbf', random_state=2, C=C, gamma=gamma).fit(x_train, y_train)
model_eva, y_proba, auc_score, fpr, tpr, cm = model_index('SVM_Train', model, model_eva, x_train, y_train)
model_eva, y_proba, auc_score, fpr, tpr, cm = model_index('SVM_Test', model, model_eva, x_test, y_test)
model_eva